# Module 3 — EHR / Clinical Feature Encoder
**MammoAssist** · Breast USG Clinical Data Processing & Feature Engineering

This notebook:
1. Loads the BrEaST-Lesions clinical CSV (21 columns, 256 cases)
2. Performs EDA — class balance, missing values, feature-target distributions
3. Engineers a fixed-size feature vector per sample (one-hot, multi-hot, ordinal, binary)
4. Defines the `EHREncoder` (lightweight MLP, sized for N≈252)
5. Saves the processed features to `ehr_processed.pkl` for the classification notebook

In [ ]:
# ============================================================
# 0) IMPORTS
# ============================================================
!pip -q install openpyxl

import os, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from scipy.stats import pointbiserialr

import torch
import torch.nn as nn

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('Imports OK ✓')

In [ ]:
# ============================================================
# 1) CONFIGURATION — update paths for your environment
# ============================================================
EXCEL_PATH = (
    "/kaggle/input/datasets/hiralisangani/breast-images-and-clinical/"
    "BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx"
)
SHEET_CLINICAL = "BrEaST-Lesions-USG clinical dat"   # sheet 1: clinical data
SHEET_DICT     = "Data Dictionary"                    # sheet 2: data dictionary

WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)
SEED = 42

In [ ]:
# ============================================================
# 2) LOAD EXCEL (two sheets: clinical data + data dictionary)
# ============================================================
df_raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_CLINICAL)
print(f"Total cases: {len(df_raw)}")
print(f"Columns ({len(df_raw.columns)}): {list(df_raw.columns)}")

# Also peek at the data dictionary sheet
df_dict = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_DICT)
print(f"\nData Dictionary ({len(df_dict)} entries):")
print(df_dict.to_string())

df_raw.head(3)

In [ ]:
# ============================================================
# 3) CLASS DISTRIBUTION & CLEANUP
# ============================================================
print("Raw class distribution:")
print(df_raw['Classification'].value_counts())

# Drop 'normal' cases (only 4 — too few to form a useful class)
df = df_raw[df_raw['Classification'].isin(['benign', 'malignant'])].copy()
df['label'] = (df['Classification'] == 'malignant').astype(int)

print(f"\nAfter dropping normals: {len(df)} cases")
print(f"  Benign:    {(df['label']==0).sum()}")
print(f"  Malignant: {(df['label']==1).sum()}")
print(f"  Imbalance ratio (benign:malignant): "
      f"1:{(df['label']==0).sum()/(df['label']==1).sum():.2f}")

fig, ax = plt.subplots(figsize=(4, 3.5))
df['Classification'].value_counts().plot.bar(
    ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='black')
ax.set_title('Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(df['Classification'].value_counts()):
    ax.text(i, v + 1, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4) MISSING VALUES ANALYSIS
# ============================================================
CLINICAL_COLS = [
    'Age', 'Tissue_composition', 'Signs', 'Symptoms',
    'Shape', 'Margin', 'Echogenicity', 'Posterior_features',
    'Halo', 'Calcifications', 'Skin_thickening', 'BIRADS'
]

def is_missing(val):
    return str(val).strip().lower() in ['not available', 'nan', '']

missing = {c: df[c].apply(is_missing).sum() for c in CLINICAL_COLS}
miss_df = pd.DataFrame({
    'Missing': missing.values(),
    'Pct': [v / len(df) * 100 for v in missing.values()]
}, index=missing.keys()).sort_values('Pct', ascending=False)

print("Missing Values Summary")
print("=" * 45)
print(miss_df.to_string())

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ['#e74c3c' if p > 20 else '#e67e22' if p > 0 else '#2ecc71'
          for p in miss_df['Pct']]
miss_df['Pct'].plot.barh(ax=ax, color=colors, edgecolor='black')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values per Clinical Feature', fontweight='bold')
ax.axvline(x=20, color='red', ls='--', alpha=0.5, label='20 % threshold')
ax.legend()
plt.tight_layout()
plt.savefig(f"{WORK_DIR}/ehr_missing_values.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5) FEATURE DISTRIBUTIONS BY CLASS
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# BIRADS
ax = axes[0, 0]
pd.crosstab(df['BIRADS'], df['Classification']).plot.bar(
    ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('BIRADS vs Class'); ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)

# Shape
ax = axes[0, 1]
pd.crosstab(df['Shape'], df['Classification']).plot.bar(
    ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Shape vs Class')
ax.tick_params(axis='x', rotation=20)

# Echogenicity
ax = axes[0, 2]
pd.crosstab(df['Echogenicity'], df['Classification']).plot.bar(
    ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Echogenicity vs Class')
ax.tick_params(axis='x', rotation=30)

# Halo
ax = axes[1, 0]
pd.crosstab(df['Halo'], df['Classification']).plot.bar(
    ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Halo vs Class')
ax.tick_params(axis='x', rotation=0)

# Posterior features
ax = axes[1, 1]
pd.crosstab(df['Posterior_features'], df['Classification']).plot.bar(
    ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Posterior Features vs Class')
ax.tick_params(axis='x', rotation=20)

# Age (numeric)
ax = axes[1, 2]
age_df = df[~df['Age'].apply(is_missing)].copy()
age_df['Age_num'] = age_df['Age'].astype(float)
for cls, c in [('benign', '#2ecc71'), ('malignant', '#e74c3c')]:
    ax.hist(age_df.loc[age_df['Classification'] == cls, 'Age_num'],
            bins=15, alpha=0.6, color=c, label=cls, edgecolor='black')
ax.set_title('Age Distribution by Class')
ax.set_xlabel('Age'); ax.legend()

plt.suptitle('Clinical Feature Distributions by Class',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{WORK_DIR}/ehr_feature_distributions.png",
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 6) BUILD VOCABULARIES (scan all unique atomic values)
# ============================================================
def extract_vocab(series, sep='&'):
    """Extract unique atomic values, ignoring 'not available' / NaN."""
    vals = set()
    for v in series:
        s = str(v).strip()
        if s.lower() in ['not available', 'nan', '']:
            continue
        for part in s.split(sep):
            vals.add(part.strip())
    return sorted(vals)

# Single-label categoricals
SHAPE_VOCAB     = extract_vocab(df['Shape'])
ECHO_VOCAB      = extract_vocab(df['Echogenicity'])
POSTERIOR_VOCAB  = extract_vocab(df['Posterior_features'])
CALC_VOCAB       = extract_vocab(df['Calcifications'])
TISSUE_VOCAB     = extract_vocab(df['Tissue_composition'])

# Multi-label categoricals  (& separated)
MARGIN_VOCAB   = extract_vocab(df['Margin'], sep='&')
SIGNS_VOCAB    = extract_vocab(df['Signs'], sep='&')
SYMPTOMS_VOCAB = extract_vocab(df['Symptoms'], sep='&')

# BIRADS ordinal mapping
BIRADS_MAP = {'1': 0, '2': 1, '3': 2, '4a': 3, '4b': 4, '4c': 5, '5': 6}

print("Vocabularies")
print("=" * 60)
for name, v in [('Shape', SHAPE_VOCAB), ('Echogenicity', ECHO_VOCAB),
                ('Posterior', POSTERIOR_VOCAB), ('Calcifications', CALC_VOCAB),
                ('Tissue', TISSUE_VOCAB), ('Margin', MARGIN_VOCAB),
                ('Signs', SIGNS_VOCAB), ('Symptoms', SYMPTOMS_VOCAB)]:
    print(f"  {name:18s} ({len(v):2d}): {v}")
print(f"  {'BIRADS':18s}  map: {BIRADS_MAP}")

In [ ]:
# ============================================================
# 7) ENCODING FUNCTIONS
# ============================================================
def encode_age(val):
    """→ (normalised_age, missing_flag)  —  2 dims."""
    if is_missing(val):
        return 0.0, 1.0
    return np.clip((float(val) - 18.0) / 82.0, 0, 1), 0.0  # 18–100

def encode_birads(val):
    """→ ordinal float in [0, 1]  —  1 dim."""
    return BIRADS_MAP.get(str(val).strip(), 3) / 6.0

def encode_onehot(val, vocab):
    """→ one-hot + extra 'unknown' dim  —  len(vocab)+1 dims."""
    vec = np.zeros(len(vocab) + 1, dtype=np.float32)
    s = str(val).strip()
    if is_missing(s):
        vec[-1] = 1.0
    elif s in vocab:
        vec[vocab.index(s)] = 1.0
    else:
        vec[-1] = 1.0
    return vec

def encode_multihot(val, vocab, sep='&'):
    """→ multi-hot + extra 'unknown' dim  —  len(vocab)+1 dims."""
    vec = np.zeros(len(vocab) + 1, dtype=np.float32)
    s = str(val).strip()
    if is_missing(s):
        vec[-1] = 1.0
        return vec
    for part in s.split(sep):
        p = part.strip()
        if p in vocab:
            vec[vocab.index(p)] = 1.0
        else:
            vec[-1] = 1.0
    return vec

def encode_binary(val):
    """yes → 1.0, anything else → 0.0  —  1 dim."""
    return 1.0 if str(val).strip().lower() == 'yes' else 0.0

print("Encoding functions defined ✓")

In [ ]:
# ============================================================
# 8) BUILD FEATURE VECTORS FOR ALL 252 SAMPLES
# ============================================================
feature_records = {}

for _, row in df.iterrows():
    cid = f"case{int(row['CaseID']):03d}"
    parts = []

    # 1) Age: normalised + missing flag  (2)
    a_norm, a_miss = encode_age(row['Age'])
    parts += [a_norm, a_miss]

    # 2) BIRADS ordinal  (1)
    parts.append(encode_birads(row['BIRADS']))

    # 3) Shape one-hot  (|V|+1)
    parts.extend(encode_onehot(row['Shape'], SHAPE_VOCAB))

    # 4) Margin multi-hot  (|V|+1)
    parts.extend(encode_multihot(row['Margin'], MARGIN_VOCAB, '&'))

    # 5) Echogenicity one-hot  (|V|+1)
    parts.extend(encode_onehot(row['Echogenicity'], ECHO_VOCAB))

    # 6) Posterior features one-hot  (|V|+1)
    parts.extend(encode_onehot(row['Posterior_features'], POSTERIOR_VOCAB))

    # 7) Halo binary  (1)
    parts.append(encode_binary(row['Halo']))

    # 8) Calcifications one-hot  (|V|+1)
    parts.extend(encode_onehot(row['Calcifications'], CALC_VOCAB))

    # 9) Skin thickening binary  (1)
    parts.append(encode_binary(row['Skin_thickening']))

    # 10) Tissue composition one-hot  (|V|+1)
    parts.extend(encode_onehot(row['Tissue_composition'], TISSUE_VOCAB))

    # 11) Signs multi-hot  (|V|+1)
    parts.extend(encode_multihot(row['Signs'], SIGNS_VOCAB, '&'))

    # 12) Symptoms multi-hot  (|V|+1)
    parts.extend(encode_multihot(row['Symptoms'], SYMPTOMS_VOCAB, '&'))

    feature_records[cid] = {
        'features': np.array(parts, dtype=np.float32),
        'label': int(row['label']),
        'classification': row['Classification'],
        'birads': str(row['BIRADS']).strip(),
    }

FEAT_DIM = feature_records[list(feature_records.keys())[0]]['features'].shape[0]
print(f"Encoded {len(feature_records)} samples  |  feature dim = {FEAT_DIM}")

In [ ]:
# ============================================================
# 9) FEATURE VECTOR BREAKDOWN
# ============================================================
idx = 0
feature_index_map = {}
dims = [
    ('Age (norm + miss)',          2),
    ('BIRADS (ordinal)',           1),
    ('Shape (one-hot)',            len(SHAPE_VOCAB) + 1),
    ('Margin (multi-hot)',         len(MARGIN_VOCAB) + 1),
    ('Echogenicity (one-hot)',     len(ECHO_VOCAB) + 1),
    ('Posterior feat. (one-hot)',  len(POSTERIOR_VOCAB) + 1),
    ('Halo (binary)',             1),
    ('Calcifications (one-hot)',  len(CALC_VOCAB) + 1),
    ('Skin thickening (binary)',  1),
    ('Tissue comp. (one-hot)',    len(TISSUE_VOCAB) + 1),
    ('Signs (multi-hot)',         len(SIGNS_VOCAB) + 1),
    ('Symptoms (multi-hot)',      len(SYMPTOMS_VOCAB) + 1),
]

print("Feature Vector Layout")
print("=" * 55)
total = 0
for name, d in dims:
    print(f"  [{idx:3d}:{idx+d:3d})  {name:30s}  ({d} dims)")
    feature_index_map[name] = (idx, idx + d)
    idx += d; total += d
print("=" * 55)
print(f"  Total: {total} dimensions")
assert total == FEAT_DIM, f"Mismatch: {total} vs {FEAT_DIM}"

In [ ]:
# ============================================================
# 10) FEATURE–TARGET CORRELATION (point-biserial)
# ============================================================
keys_sorted = sorted(feature_records.keys())
X = np.stack([feature_records[k]['features'] for k in keys_sorted])
y = np.array([feature_records[k]['label'] for k in keys_sorted])

corrs = []
for i in range(X.shape[1]):
    if X[:, i].std() > 0:
        r, _ = pointbiserialr(y, X[:, i])
        corrs.append(r)
    else:
        corrs.append(0.0)

# Map dim index → feature group name
def idx_to_name(fi):
    for name, (s, e) in feature_index_map.items():
        if s <= fi < e:
            return f"{name}[{fi - s}]"
    return f"dim_{fi}"

top_idx = np.argsort(np.abs(corrs))[::-1][:20]
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#e74c3c' if corrs[i] > 0 else '#3498db' for i in top_idx]
ax.barh(range(len(top_idx)), [corrs[i] for i in top_idx], color=colors)
ax.set_yticks(range(len(top_idx)))
ax.set_yticklabels([idx_to_name(i) for i in top_idx], fontsize=9)
ax.set_xlabel('Point-biserial r with malignancy')
ax.set_title('Top-20 Feature–Target Correlations', fontweight='bold')
ax.axvline(0, color='black', lw=0.5)
plt.tight_layout()
plt.savefig(f"{WORK_DIR}/ehr_feature_correlations.png",
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 11) EHR ENCODER — PyTorch Module
#     Lightweight MLP designed for small datasets (N ≈ 252)
# ============================================================
class EHREncoder(nn.Module):
    """
    Single-hidden-layer MLP for pre-processed clinical feature vectors.
    Compact by design: only ~8 k params with default settings.
    """
    def __init__(self, input_dim, hidden_dim=64, output_dim=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        """x: (B, input_dim) → (B, output_dim)"""
        return self.net(x)

# Quick sanity check
ehr_enc = EHREncoder(FEAT_DIM, 64, 64)
dummy = torch.randn(4, FEAT_DIM)
out = ehr_enc(dummy)
n_params = sum(p.numel() for p in ehr_enc.parameters())
print(f"EHREncoder  ({FEAT_DIM},) → ({out.shape[1]},)")
print(f"Trainable params: {n_params:,}")

In [ ]:
# ============================================================
# 12) SAVE PROCESSED FEATURES & VOCABS
# ============================================================
ehr_data = {
    'feature_records': feature_records,   # {case_id: {features, label, ...}}
    'feat_dim':        FEAT_DIM,
    'feature_index_map': feature_index_map,
    'vocabs': {
        'SHAPE': SHAPE_VOCAB,
        'ECHO': ECHO_VOCAB,
        'POSTERIOR': POSTERIOR_VOCAB,
        'CALC': CALC_VOCAB,
        'TISSUE': TISSUE_VOCAB,
        'MARGIN': MARGIN_VOCAB,
        'SIGNS': SIGNS_VOCAB,
        'SYMPTOMS': SYMPTOMS_VOCAB,
        'BIRADS_MAP': BIRADS_MAP,
    }
}

save_path = os.path.join(WORK_DIR, 'ehr_processed.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(ehr_data, f)

labels = [r['label'] for r in feature_records.values()]
print(f"Saved → {save_path}")
print(f"  Samples:   {len(feature_records)}")
print(f"  Feat dim:  {FEAT_DIM}")
print(f"  Labels:    {Counter(labels)}")
print("\nModule 3 complete ✓  →  load this file in the classification notebook")